![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## GET COMMON DATA

In [10]:
import pandas as pd
import pickle

# ===== LOAD DATA =====
print("="*80)
print("LOADING DATA")
print("="*80)

# Load 24/7 dataset (should be .pkl not .parquet)
with open('merged_common_selected.pkl', 'rb') as f:
    merged_common_final = pickle.load(f)

print(f"\nDataset: {merged_common_final.shape}")

# Load crypto (24/7) features only
with open('features_all.pkl', 'rb') as f:
    top_features_mi = pickle.load(f)

print(f"Features: {len(top_features_mi)} (MARKET only)")

# Define all targets
target_cols = [
    'target_n3',
    'target_n3_001',
    'target_sharpe_n3_10',
    'target_sharpe_n3_20',
    'target_sortino_n3_05',
    'target_sortino_n3_15',
    'target_upr_n3_13'
]

print(f"Targets: {len(target_cols)}")

print(f"\nFinal features: {len(top_features_mi)}")
print(f"\n✓ Ready for modeling!")
print(f"  X: {len(top_features_mi)} features")
print(f"  y: {len(target_cols)} targets")

In [11]:
# Check for infinities in the feature columns
X_features = merged_common_final[top_features_mi]

print(f"\n3. TOTAL INFINITY CHECK (both positive and negative)")
total_inf = np.isinf(X_features).sum().sum()
print(f"   Total infinities: {total_inf:,}")
print(f"   Percentage of all values: {total_inf / (X_features.shape[0] * X_features.shape[1]) * 100:.4f}%")
print(f"   Rows with ANY infinity: {np.isinf(X_features).any(axis=1).sum():,} ({np.isinf(X_features).any(axis=1).sum()/len(X_features)*100:.2f}%)")

# Simple one-liner
merged_common_final = merged_common_final.replace([np.inf, -np.inf], np.nan)

### Feature importance

In [12]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, f1_score, matthews_corrcoef
from sklearn.model_selection import TimeSeriesSplit

# ===== STEP 1: TIME SERIES CV FOR PERFORMANCE EVALUATION =====
print("="*80)
print("STEP 1: TIME SERIES CROSS-VALIDATION - PERFORMANCE STUDY")
print("="*80)

X_all = merged_common_final[top_features_mi]

all_fold_results = {}
all_results = {}

n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

for target_name in target_cols:
    print(f"Processing {target_name}...", end=" ")
    
    y_all = merged_common_final[target_name]
    fold_metrics = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_all), 1):
        X_train, X_val = X_all.iloc[train_idx], X_all.iloc[val_idx]
        y_train, y_val = y_all.iloc[train_idx], y_all.iloc[val_idx]
        
        # Calculate scale_pos_weight for imbalanced classes
        n_neg = (y_train == 0).sum()
        n_pos = (y_train == 1).sum()
        
        if n_pos == 0:
            continue
            
        scale_pos_weight = n_neg / n_pos
        
        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.02,
            min_child_weight=20,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=2.0,
            reg_lambda=5.0,
            scale_pos_weight=scale_pos_weight,
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1,
        )
        
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        
        # Get predictions
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        y_val_pred_proba = model.predict_proba(X_val)[:, 1]
        
        # Calculate metrics
        train_acc = accuracy_score(y_train, y_train_pred)
        val_acc = accuracy_score(y_val, y_val_pred)
        val_roc_auc = roc_auc_score(y_val, y_val_pred_proba)
        val_pr_auc = average_precision_score(y_val, y_val_pred_proba)
        val_f1 = f1_score(y_val, y_val_pred)
        val_mcc = matthews_corrcoef(y_val, y_val_pred)
        
        fold_metrics.append({
            'fold': fold_idx,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'overfit_gap': train_acc - val_acc,
            'roc_auc': val_roc_auc,
            'pr_auc': val_pr_auc,
            'f1': val_f1,
            'mcc': val_mcc
        })
    
    # Aggregate fold results
    fold_df = pd.DataFrame(fold_metrics)
    
    all_fold_results[target_name] = fold_df
    all_results[target_name] = {
        'pos_rate': y_all.mean(),
        'mean_train_acc': fold_df['train_acc'].mean(),
        'mean_val_acc': fold_df['val_acc'].mean(),
        'std_val_acc': fold_df['val_acc'].std(),
        'mean_overfit_gap': fold_df['overfit_gap'].mean(),
        'mean_roc_auc': fold_df['roc_auc'].mean(),
        'std_roc_auc': fold_df['roc_auc'].std(),
        'mean_pr_auc': fold_df['pr_auc'].mean(),
        'mean_f1': fold_df['f1'].mean(),
        'mean_mcc': fold_df['mcc'].mean(),
        'std_mcc': fold_df['mcc'].std(),
    }
    
    print(f"✓ ROC: {all_results[target_name]['mean_roc_auc']:.4f}")

print("\n" + "="*80)
print("SUMMARY: ALL TARGETS")
print("="*80)

summary_df = pd.DataFrame(all_results).T
summary_df = summary_df.sort_values('mean_roc_auc', ascending=False)

# Display key metrics
display_cols = ['pos_rate', 'mean_roc_auc', 'std_roc_auc', 'mean_mcc', 
                'mean_overfit_gap', 'mean_val_acc']
print(summary_df[display_cols].round(4).to_string())

# ===== STEP 2: TRAIN MODELS ON ALL DATA FOR FEATURE IMPORTANCE =====
print("="*80)
print("STEP 2: TRAINING MODELS ON ALL DATA (FOR FEATURE IMPORTANCE ONLY)")
print("="*80)

all_models = {}

for target_name in target_cols:
    print(f"Training {target_name}...", end=" ")
    
    y_all = merged_common_final[target_name]
    
    n_neg = (y_all == 0).sum()
    n_pos = (y_all == 1).sum()
    scale_pos_weight = n_neg / n_pos
    
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.02,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=2.0,
        reg_lambda=5.0,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
    )
    
    model.fit(X_all, y_all, verbose=False)
    all_models[target_name] = model
    print("✓")

print("\n✓ All models trained on full dataset")

#### XGBOOST Feature Importance: TimeSeriesSplit 10 filds, built-in XGBoost Importance.

In [13]:
# ===== SELECTED TARGETS FOR FEATURE SELECTION (OPTION C) =====
selected_targets = [
    'target_n3_001',           # Primary: 0.5616 ROC, 0.0801 MCC
    'target_sharpe_n3_20',     # Stable risk-adjusted: 0.5403 ROC
    'target_sortino_n3_15',    # Downside-focused: 0.5339 ROC
]

print("\n" + "="*80)
print("SELECTED TARGETS FOR FEATURE SELECTION")
print("="*80)
for target in selected_targets:
    print(f"  ✓ {target}")
print(f"\nTotal: {len(selected_targets)} targets")

# Filter models to only selected targets
selected_models = {k: v for k, v in all_models.items() if k in selected_targets}

# ===== XGBOOST GAIN IMPORTANCE ACROSS SELECTED TARGETS =====
print("\n" + "="*80)
print(f"XGBOOST GAIN IMPORTANCE (AGGREGATED ACROSS {len(selected_targets)} TARGETS)")
print("="*80)

# Collect importance from selected models only
all_importance = {}

for target_name, model in selected_models.items():
    importances = model.feature_importances_
    for feature, importance in zip(X_all.columns, importances):
        if feature not in all_importance:
            all_importance[feature] = []
        all_importance[feature].append(importance)

# Aggregate: mean, max, and count of non-zero
gain_importance_summary = []
for feature, importance_list in all_importance.items():
    gain_importance_summary.append({
        'feature': feature,
        'mean_importance': np.mean(importance_list),
        'max_importance': np.max(importance_list),
        'min_importance': np.min(importance_list),
        'std_importance': np.std(importance_list),
        'n_targets_nonzero': sum(1 for imp in importance_list if imp > 0),
        'n_targets_gt_0.001': sum(1 for imp in importance_list if imp > 0.001)
    })

gain_importance_df = pd.DataFrame(gain_importance_summary)
gain_importance_df = gain_importance_df.sort_values('mean_importance', ascending=False)

print(f"\nTotal features: {len(gain_importance_df)}")
print(f"Features used in all {len(selected_targets)} targets: {(gain_importance_df['n_targets_nonzero'] == len(selected_targets)).sum()}")
print(f"Features with mean importance > 0.001: {(gain_importance_df['mean_importance'] > 0.001).sum()}")

print(f"\nTop 50 features (by mean importance across {len(selected_targets)} targets):")
print(gain_importance_df.head(50).to_string(index=False))

# Save for later analysis
gain_importance_df.to_csv('feature_importance_gain_aggregated.csv', index=False)
print("\n✓ Saved aggregated importance to 'feature_importance_gain_aggregated.csv'")

SHAP STILL DELETES NANS

In [14]:
import shap
import numpy as np

# ===== STEP 3: SHAP FEATURE IMPORTANCE ACROSS SELECTED TARGETS =====
print("\n" + "="*80)
print(f"STEP 3: SHAP FEATURE IMPORTANCE (AGGREGATED ACROSS {len(selected_targets)} TARGETS)")
print("="*80)

# Fixed sample size
sample_size = 7500
total_rows = len(X_all)
print(f"Total dataset size: {total_rows:,} rows")
print(f"Using {sample_size:,} samples ({sample_size/total_rows*100:.1f}% of data)")

# Stratified sampling across time for regime diversity
n_chunks = 10
chunk_size = len(X_all) // n_chunks
sample_per_chunk = sample_size // n_chunks

sampled_indices = []
for i in range(n_chunks):
    chunk_start = i * chunk_size
    chunk_end = (i + 1) * chunk_size if i < n_chunks - 1 else len(X_all)
    chunk_indices = np.arange(chunk_start, chunk_end)
    sampled_chunk = np.random.choice(chunk_indices, size=sample_per_chunk, replace=False)
    sampled_indices.extend(sampled_chunk)

X_sample = X_all.iloc[sampled_indices]
print(f"Sampled evenly across {n_chunks} time periods for regime diversity")

# Collect SHAP values from selected models only
all_shap_importance = {}

for target_name, model in selected_models.items():
    print(f"  Computing SHAP for {target_name}...", end=" ")
    
    # Create explainer
    explainer = shap.TreeExplainer(model)
    
    # Compute SHAP values
    shap_values = explainer.shap_values(X_sample)
    
    # Get mean absolute SHAP values for this target
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    print(f"✓ (max SHAP: {mean_abs_shap.max():.4f})")
    
    for feature, importance in zip(X_sample.columns, mean_abs_shap):
        if feature not in all_shap_importance:
            all_shap_importance[feature] = []
        all_shap_importance[feature].append(importance)

# Aggregate SHAP importance
shap_importance_summary = []
for feature, importance_list in all_shap_importance.items():
    shap_importance_summary.append({
        'feature': feature,
        'mean_importance': np.mean(importance_list),
        'max_importance': np.max(importance_list),
        'min_importance': np.min(importance_list),
        'std_importance': np.std(importance_list),
        'n_targets_gt_0': sum(1 for imp in importance_list if imp > 0),
        'n_targets_gt_0.01': sum(1 for imp in importance_list if imp > 0.01)
    })

shap_importance_df = pd.DataFrame(shap_importance_summary)
shap_importance_df = shap_importance_df.sort_values('mean_importance', ascending=False)

print("\n" + "="*80)
print("AGGREGATED SHAP IMPORTANCE")
print("="*80)
print(f"Total features: {len(shap_importance_df)}")
print(f"Features used in all {len(selected_targets)} targets: {(shap_importance_df['n_targets_gt_0'] == len(selected_targets)).sum()}")
print(f"Features with mean importance > 0.01: {(shap_importance_df['mean_importance'] > 0.01).sum()}")

print(f"\nTop 50 features (by mean SHAP importance across {len(selected_targets)} targets):")
print(shap_importance_df.head(50).to_string(index=False))

# Save results
shap_importance_df.to_csv('feature_importance_shap_aggregated.csv', index=False)
print(f"\n✓ Saved aggregated SHAP importance to 'feature_importance_shap_aggregated.csv'")
print(f"✓ Computation used {sample_size:,} samples for stability")

In [15]:
import matplotlib.pyplot as plt

# ===== OVERLAP ANALYSIS: SHAP vs GAIN (AGGREGATED) =====
print("\n" + "="*80)
print("OVERLAP ANALYSIS: SHAP vs GAIN (AGGREGATED ACROSS 6 TARGETS)")
print("="*80)

# Calculate overlap counts at different thresholds
thresholds = [10, 25, 50, 75, 100, 125, 150, 175, 200]
overlap_counts = []
overlap_percentages = []

for n in thresholds:
    top_n_gain = set(gain_importance_df.head(n)['feature'])
    top_n_shap = set(shap_importance_df.head(n)['feature'])
    overlap = top_n_gain & top_n_shap
    
    overlap_count = len(overlap)
    overlap_pct = (overlap_count / n) * 100
    
    overlap_counts.append(overlap_count)
    overlap_percentages.append(overlap_pct)
    
    print(f"Top {n:3d}: {overlap_count:3d} features overlap ({overlap_pct:.1f}%)")

# Create two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Absolute overlap count
ax1.plot(thresholds, overlap_counts, marker='o', linewidth=3, markersize=10, color='#2E86AB')
ax1.plot(thresholds, thresholds, linestyle='--', color='gray', alpha=0.5, label='Perfect agreement')
ax1.set_xlabel('Top N Features', fontsize=13)
ax1.set_ylabel('Number of Overlapping Features', fontsize=13)
ax1.set_title('SHAP vs Gain: Absolute Feature Overlap\n(Aggregated across 6 targets)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Add count labels
for x, y in zip(thresholds, overlap_counts):
    ax1.text(x, y + 5, f'{y}', ha='center', fontsize=10, fontweight='bold')

# Plot 2: Percentage overlap
ax2.plot(thresholds, overlap_percentages, marker='s', linewidth=3, markersize=10, color='#A23B72')
ax2.axhline(y=70, color='red', linestyle='--', alpha=0.7, label='70% threshold', linewidth=2)
ax2.axhline(y=50, color='orange', linestyle='--', alpha=0.7, label='50% threshold', linewidth=2)
ax2.set_xlabel('Top N Features', fontsize=13)
ax2.set_ylabel('Overlap Percentage (%)', fontsize=13)
ax2.set_title('SHAP vs Gain: Percentage Agreement\n(Aggregated across 6 targets)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()
ax2.set_ylim([0, 105])

# Add percentage labels
for x, y in zip(thresholds, overlap_percentages):
    ax2.text(x, y + 2, f'{y:.0f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('shap_vs_gain_overlap_aggregated.png', dpi=300, bbox_inches='tight')
plt.show()


In [16]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, matthews_corrcoef
from sklearn.model_selection import TimeSeriesSplit

# ===== SYSTEMATIC CONSENSUS FEATURE TESTING WITH TIME SERIES CV =====
print("\n" + "="*80)
print("TESTING DIFFERENT CONSENSUS SIZES (INTERSECTION) - TIME SERIES CV")
print("="*80)

# Test different top-N values for consensus
consensus_ns = [5, 10, 15, 20, 30, 50, 70, 85]
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

consensus_results = []

for n in consensus_ns:
    # Get consensus features
    top_n_shap = set(shap_importance_df.head(n)['feature'])
    top_n_gain = set(gain_importance_df.head(n)['feature'])
    consensus_features = list(top_n_shap & top_n_gain)
    
    n_consensus = len(consensus_features)
    
    if n_consensus == 0:
        continue
    
    print(f"\nTop {n} → {n_consensus} consensus features ({n_consensus/n*100:.0f}%)", end=" ")
    
    # Prepare data
    X_all_test = merged_common_final[consensus_features]
    
    # Train all targets with CV
    target_results = []
    
    for target_name in target_cols:
        y_all = merged_common_final[target_name]
        
        fold_metrics = []
        
        for train_idx, val_idx in tscv.split(X_all_test):
            X_train, X_val = X_all_test.iloc[train_idx], X_all_test.iloc[val_idx]
            y_train, y_val = y_all.iloc[train_idx], y_all.iloc[val_idx]
            
            # Scale pos weight
            n_neg = (y_train == 0).sum()
            n_pos = (y_train == 1).sum()
            if n_pos == 0:
                continue
            scale_pos_weight = n_neg / n_pos
            
            # Train
            model = xgb.XGBClassifier(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.02,
                min_child_weight=20,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=2.0,
                reg_lambda=5.0,
                scale_pos_weight=scale_pos_weight,
                eval_metric='logloss',
                random_state=42,
                n_jobs=-1,
            )
            
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
            
            # Evaluate
            y_val_pred = model.predict(X_val)
            y_val_pred_proba = model.predict_proba(X_val)[:, 1]
            
            train_acc = accuracy_score(y_train, model.predict(X_train))
            val_acc = accuracy_score(y_val, y_val_pred)
            val_roc_auc = roc_auc_score(y_val, y_val_pred_proba)
            val_mcc = matthews_corrcoef(y_val, y_val_pred)
            
            fold_metrics.append({
                'train_acc': train_acc,
                'val_acc': val_acc,
                'overfit_gap': train_acc - val_acc,
                'roc_auc': val_roc_auc,
                'mcc': val_mcc
            })
        
        # Average across folds for this target
        fold_df = pd.DataFrame(fold_metrics)
        target_results.append({
            'target': target_name,
            'train_acc': fold_df['train_acc'].mean(),
            'val_acc': fold_df['val_acc'].mean(),
            'overfit_gap': fold_df['overfit_gap'].mean(),
            'roc_auc': fold_df['roc_auc'].mean(),
            'mcc': fold_df['mcc'].mean(),
            'roc_auc_std': fold_df['roc_auc'].std()
        })
    
    # Aggregate across all targets
    target_df = pd.DataFrame(target_results)
    
    avg_roc = target_df['roc_auc'].mean()
    avg_overfit = target_df['overfit_gap'].mean()
    avg_mcc = target_df['mcc'].mean()
    
    print(f"| ROC: {avg_roc:.4f} | Overfit: {avg_overfit:.4f} | MCC: {avg_mcc:.4f}")
    
    consensus_results.append({
        'n': n,
        'n_consensus': n_consensus,
        'consensus_pct': (n_consensus / n) * 100,
        'avg_val_acc': target_df['val_acc'].mean(),
        'avg_roc_auc': avg_roc,
        'avg_overfit': avg_overfit,
        'avg_mcc': avg_mcc,
        'std_roc_auc': target_df['roc_auc'].std(),
        'min_roc_auc': target_df['roc_auc'].min(),
        'max_roc_auc': target_df['roc_auc'].max(),
    })

# Create results DataFrame
consensus_df = pd.DataFrame(consensus_results)

print("\n" + "="*80)
print("SUMMARY: CONSENSUS PERFORMANCE BY N")
print("="*80)
print(consensus_df[['n', 'n_consensus', 'avg_roc_auc', 'avg_overfit', 'avg_mcc']].to_string(index=False))

# Find best N
best_roc_idx = consensus_df['avg_roc_auc'].idxmax()
best_n = consensus_df.loc[best_roc_idx, 'n']
best_n_consensus = consensus_df.loc[best_roc_idx, 'n_consensus']
best_roc = consensus_df.loc[best_roc_idx, 'avg_roc_auc']

print("\n" + "="*80)
print("BEST CONSENSUS")
print("="*80)
print(f"Top {int(best_n)} → {int(best_n_consensus)} features | ROC: {best_roc:.4f} | Overfit: {consensus_df.loc[best_roc_idx, 'avg_overfit']:.4f} | MCC: {consensus_df.loc[best_roc_idx, 'avg_mcc']:.4f}")

# ===== VISUALIZATION =====
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: ROC-AUC vs N
ax1 = axes[0, 0]
ax1.plot(consensus_df['n'], consensus_df['avg_roc_auc'], marker='o', linewidth=3, 
         markersize=10, color='#2E86AB', label='Avg ROC-AUC')
ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random (0.5)')
ax1.scatter([best_n], [best_roc], color='red', s=200, marker='*', 
            zorder=5, label=f'Best: n={int(best_n)}')
ax1.set_xlabel('Top N for Consensus', fontsize=12)
ax1.set_ylabel('Average ROC-AUC', fontsize=12)
ax1.set_title('Performance vs Consensus Size (Time Series CV)', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Number of consensus features
ax2 = axes[0, 1]
ax2.plot(consensus_df['n'], consensus_df['n_consensus'], marker='s', linewidth=3, 
         markersize=10, color='#A23B72')
ax2.plot(consensus_df['n'], consensus_df['n'], linestyle='--', color='gray', 
         alpha=0.5, label='Perfect agreement')
ax2.set_xlabel('Top N for Consensus', fontsize=12)
ax2.set_ylabel('Number of Consensus Features', fontsize=12)
ax2.set_title('Consensus Features vs N', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

for x, y in zip(consensus_df['n'], consensus_df['n_consensus']):
    ax2.text(x, y + 2, f'{int(y)}', ha='center', fontsize=9, fontweight='bold')

# Plot 3: Overfit Gap
ax3 = axes[1, 0]
ax3.plot(consensus_df['n'], consensus_df['avg_overfit'], marker='^', linewidth=3, 
         markersize=10, color='#F18F01')
ax3.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='No overfit')
ax3.set_xlabel('Top N for Consensus', fontsize=12)
ax3.set_ylabel('Average Overfit Gap', fontsize=12)
ax3.set_title('Overfitting vs Consensus Size', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: MCC
ax4 = axes[1, 1]
ax4.plot(consensus_df['n'], consensus_df['avg_mcc'], marker='D', linewidth=3, 
         markersize=10, color='#6A994E')
ax4.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='No correlation')
ax4.set_xlabel('Top N for Consensus', fontsize=12)
ax4.set_ylabel('Average MCC', fontsize=12)
ax4.set_title('Matthews Correlation Coefficient vs N', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('consensus_feature_testing.png', dpi=300, bbox_inches='tight')
plt.show()

consensus_df.to_csv('consensus_testing_results.csv', index=False)
print("\n✓ Saved to 'consensus_testing_results.csv' and 'consensus_feature_testing.png'")

In [20]:
# ===== FINAL FEATURE SELECTION: CONSENSUS FROM TOP 25 =====
print("\n" + "="*80)
print("FINAL FEATURE SELECTION: CONSENSUS FROM TOP 25")
print("="*80)

# Get consensus features from top 25 (optimal from systematic testing)
top_n = 40

top_25_shap = set(shap_importance_df.head(top_n)['feature'])
top_25_gain = set(gain_importance_df.head(top_n)['feature'])

# Intersection: Features that appear in BOTH top 25 lists
consensus_features = list(top_25_shap & top_25_gain)

print(f"\nConsensus: {len(consensus_features)} features appear in BOTH top {top_n}")
print(f"Agreement: {len(consensus_features)}/{top_n} = {len(consensus_features)/top_n*100:.1f}%")

# Show the consensus features (sorted alphabetically for clarity)
print("\n" + "="*80)
print(f"THE {len(consensus_features)} CONSENSUS FEATURES")
print("="*80)

for i, feat in enumerate(sorted(consensus_features), 1):
    print(f"  {i:2d}. {feat}")

# Create final dataset with selected features + ALL targets
final_features = consensus_features
cols_to_keep = final_features + target_cols

merged_common_final_selected = merged_common_final[cols_to_keep]

print("\n" + "="*80)
print("FINAL DATASET")
print("="*80)
print(f"Shape:    {merged_common_final_selected.shape}")
print(f"Features: {len(final_features)}")
print(f"Targets:  {len(target_cols)}")

print(f"\nTargets:")
for i, tgt in enumerate(target_cols, 1):
    print(f"  {i}. {tgt}")

# Save the final feature list
with open('final_consensus_features.txt', 'w') as f:
    f.write("FINAL CONSENSUS FEATURES (Top 25 Intersection)\n")
    f.write("="*80 + "\n")
    f.write(f"Total: {len(final_features)} features\n")
    f.write(f"Selected from top {top_n} of both SHAP and Gain importance\n\n")
    for i, feat in enumerate(sorted(final_features), 1):
        f.write(f"{i:2d}. {feat}\n")

print("\n✓ Saved feature list to 'final_consensus_features.txt'")

In [21]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, matthews_corrcoef, f1_score, precision_score, recall_score
from sklearn.model_selection import TimeSeriesSplit

# ===== FINAL TRAINING WITH CONSENSUS FEATURES - TIME SERIES CV =====
print("\n" + "="*80)
print("FINAL TRAINING: ALL 6 TARGETS WITH CONSENSUS FEATURES - TIME SERIES CV")
print("="*80)

X_all = merged_common_final_selected[final_features]

print(f"Dataset: {len(X_all):,} rows × {len(final_features)} features")

n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

# Store results
all_results = {}

for target_name in target_cols:
    print(f"\n{target_name}:", end=" ")
    
    y_all = merged_common_final_selected[target_name]
    
    fold_metrics = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_all), 1):
        X_train, X_val = X_all.iloc[train_idx], X_all.iloc[val_idx]
        y_train, y_val = y_all.iloc[train_idx], y_all.iloc[val_idx]
        
        # Scale pos weight
        n_neg = (y_train == 0).sum()
        n_pos = (y_train == 1).sum()
        if n_pos == 0:
            continue
        scale_pos_weight = n_neg / n_pos
        
        # Train
        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.02,
            min_child_weight=20,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=2.0,
            reg_lambda=5.0,
            scale_pos_weight=scale_pos_weight,
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1,
        )
        
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        
        # Evaluate
        y_val_pred_proba = model.predict_proba(X_val)[:, 1]
        y_val_pred = model.predict(X_val)
        
        train_acc = accuracy_score(y_train, model.predict(X_train))
        val_acc = accuracy_score(y_val, y_val_pred)
        val_roc_auc = roc_auc_score(y_val, y_val_pred_proba)
        val_mcc = matthews_corrcoef(y_val, y_val_pred)
        val_f1 = f1_score(y_val, y_val_pred)
        val_precision = precision_score(y_val, y_val_pred, zero_division=0)
        val_recall = recall_score(y_val, y_val_pred)
        
        fold_metrics.append({
            'fold': fold_idx,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'roc_auc': val_roc_auc,
            'f1': val_f1,
            'precision': val_precision,
            'recall': val_recall,
            'mcc': val_mcc,
            'overfit': train_acc - val_acc
        })
    
    # Average across folds
    fold_df = pd.DataFrame(fold_metrics)
    
    print(f"ROC: {fold_df['roc_auc'].mean():.4f}±{fold_df['roc_auc'].std():.4f} | "
          f"MCC: {fold_df['mcc'].mean():.4f}±{fold_df['mcc'].std():.4f} | "
          f"F1: {fold_df['f1'].mean():.4f}")
    
    all_results[target_name] = {
        'val_acc_mean': fold_df['val_acc'].mean(),
        'val_acc_std': fold_df['val_acc'].std(),
        'roc_auc_mean': fold_df['roc_auc'].mean(),
        'roc_auc_std': fold_df['roc_auc'].std(),
        'f1_mean': fold_df['f1'].mean(),
        'f1_std': fold_df['f1'].std(),
        'precision_mean': fold_df['precision'].mean(),
        'precision_std': fold_df['precision'].std(),
        'recall_mean': fold_df['recall'].mean(),
        'recall_std': fold_df['recall'].std(),
        'mcc_mean': fold_df['mcc'].mean(),
        'mcc_std': fold_df['mcc'].std(),
        'overfit_mean': fold_df['overfit'].mean(),
        'overfit_std': fold_df['overfit'].std()
    }

# Summary
print("\n" + "="*80)
print("SUMMARY: ALL TARGETS (MEAN ± STD ACROSS FOLDS)")
print("="*80)

summary_df = pd.DataFrame(all_results).T
display_df = summary_df[['roc_auc_mean', 'roc_auc_std', 'mcc_mean', 'mcc_std', 
                          'f1_mean', 'precision_mean', 'recall_mean', 'overfit_mean']]
print(display_df.round(4).to_string())

print("\n" + "="*80)
print("OVERALL AVERAGES")
print("="*80)
print(f"Avg ROC-AUC:     {summary_df['roc_auc_mean'].mean():.4f} (stability: ±{summary_df['roc_auc_std'].mean():.4f})")
print(f"Avg F1:          {summary_df['f1_mean'].mean():.4f} (stability: ±{summary_df['f1_std'].mean():.4f})")
print(f"Avg Precision:   {summary_df['precision_mean'].mean():.4f}")
print(f"Avg Recall:      {summary_df['recall_mean'].mean():.4f}")
print(f"Avg MCC:         {summary_df['mcc_mean'].mean():.4f} (stability: ±{summary_df['mcc_std'].mean():.4f})")
print(f"Avg Overfit gap: {summary_df['overfit_mean'].mean():.4f}")

print(f"\n✓ Training complete with {len(final_features)} consensus features across {n_splits} time series folds!")

In [19]:
import pickle

# Save dataframe as pickle
merged_common_final_selected.to_pickle('merged_common_final_selected.pkl')

# Save feature list as pickle
with open('final_features.pkl', 'wb') as f:
    pickle.dump(final_features, f)

# Save target_cols
with open('target_cols.pkl', 'wb') as f:
    pickle.dump(target_cols, f)

print("✓ Saved target_cols.pkl")

print("✓ Saved merged_common_final_selected.pkl")
print("✓ Saved final_features.pkl")